# Tugas Personal ke-1 — Computational Statistics (Week 2)

**LO1:** Menjelaskan metode Komputasi Statistik (random number generation, simulasi Monte Carlo, optimasi numerik) yang penting dalam menyelesaikan permasalahan statistik kompleks.

**Nama:** _(isi nama Anda)_  
**NIM:** _(isi NIM Anda)_

> **Catatan Transparansi AI (AI-Assisted):**
> Draf awal jawaban dan kode pada notebook ini disusun dengan bantuan AI (Claude, Anthropic) untuk membantu merangkum konsep dari materi kuliah, menyusun struktur penjelasan, dan menulis kode simulasi (Bootstrap, Metropolis-Hastings, perbandingan iteratif/rekursif). Sebelum dikumpulkan, jawaban telah saya baca ulang, saya sesuaikan dengan pemahaman saya, dan saya lengkapi dengan identitas (nama/NIM) serta refleksi pribadi. Saya tetap bertanggung jawab penuh atas keaslian dan ketepatan isi tugas ini.

Notebook ini dibagi menjadi 3 bagian sesuai soal. Semua bagian menggunakan library yang sama (di-import satu kali di sel berikutnya) sehingga notebook ini bisa dijalankan berurutan dari atas ke bawah di Google Colab tanpa error.

In [ ]:
# Import library yang dipakai di seluruh notebook (Soal 1, 2, dan 3)
import numpy as np
import matplotlib.pyplot as plt
import time

np.random.seed(42)  # agar hasil random dapat direproduksi

---
## Soal 1 [30 poin] — Perkembangan PC dan Cakupan Statistik Modern

**Pertanyaan:** Jelaskan dengan contoh nyata bagaimana perkembangan komputer pribadi (PC) mempengaruhi peran dan cakupan statistik dalam dunia industri dan penelitian modern.

### Jawaban

Sebelum era PC (sebelum ~1980-an), analisis statistik sangat dibatasi oleh keterbatasan alat hitung: statistisi bergantung pada kalkulator mekanik, mesin hitung, dan tabel-tabel statistik baku (tabel z, tabel t, tabel chi-square). Karena keterbatasan ini, metode yang dipakai harus memiliki **solusi analitik tertutup (closed-form)** — misalnya uji-t, regresi linear sederhana, atau pendekatan distribusi normal lewat Teorema Limit Pusat — karena perhitungan manual untuk metode yang lebih rumit (apalagi berulang ribuan kali) tidak realistis dilakukan oleh manusia.

Munculnya PC yang murah dan bertenaga pada tahun 1980-an mengubah hal ini secara fundamental. Statistisi tidak lagi dibatasi oleh "bisa dihitung tangan atau tidak", melainkan oleh "bisa disimulasikan oleh komputer atau tidak". Ini melahirkan bidang **Statistik Komputasional**. Contoh paling nyata adalah metode **Bootstrap** yang diperkenalkan oleh Bradley Efron (1979): alih-alih menurunkan rumus analitik untuk standard error atau interval kepercayaan suatu statistik (misalnya median, korelasi, atau rasio yang rumus sebaran samplingnya sulit diturunkan), Bootstrap cukup **menyampel ulang (resampling) data yang sudah ada dengan pengembalian**, ribuan kali, lalu menghitung sebaran empiris dari statistik tersebut. Ini hanya mungkin dilakukan karena adanya daya komputasi PC untuk mengulang proses resampling secara masif dan cepat.

Dampaknya ke industri dan penelitian modern sangat luas:

- **Industri keuangan:** simulasi Monte Carlo dipakai untuk menghitung *Value at Risk* (VaR) dan menentukan harga instrumen derivatif, di mana solusi analitiknya tidak ada atau terlalu kompleks.
- **Industri farmasi/klinis:** Bootstrap dipakai untuk membangun interval kepercayaan efek obat dari uji klinis dengan ukuran sampel kecil, tanpa harus mengasumsikan bentuk distribusi data yang kaku.
- **Machine learning modern:** teknik *bagging* (bootstrap aggregating) yang menjadi dasar Random Forest langsung mengadopsi ide resampling Bootstrap.
- **Penelitian sains data:** peneliti non-statistisi (biologi, sosial, teknik) kini bisa melakukan inferensi statistik yang canggih tanpa harus menguasai turunan matematis rumit, karena PC menyediakan alat *point-and-click*/skrip untuk mensimulasikannya.

Singkatnya, PC menggeser peran statistik dari "ilmu menurunkan rumus tertutup" menjadi "ilmu mensimulasikan dan mengaproksimasi", sehingga cakupannya meluas ke masalah-masalah yang jauh lebih kompleks dan berdimensi tinggi.

### Demonstrasi Kode: Bootstrap untuk Estimasi Interval Kepercayaan

In [ ]:
# --- SOAL 1: Simulasi Bootstrap ---
# Contoh kasus: waktu respon layanan pelanggan (dalam detik) dari 25 sampel pelanggan.
# Kita ingin mengestimasi rata-rata waktu respon beserta interval kepercayaan 95%-nya,
# TANPA mengasumsikan data berdistribusi normal (kasus umum di industri nyata).

data_layanan = np.array([
    12.1, 9.8, 15.3, 11.0, 8.7, 20.5, 13.2, 10.1, 9.9, 14.7,
    22.0, 11.5, 10.8, 13.9, 9.2, 16.4, 12.8, 10.4, 25.1, 11.9,
    13.6, 9.5, 14.0, 12.3, 10.9
])

n = len(data_layanan)
n_bootstrap = 10000  # jumlah resampling -- baru mungkin dilakukan berkat daya komputasi PC

bootstrap_means = np.empty(n_bootstrap)
for i in range(n_bootstrap):
    sample_resample = np.random.choice(data_layanan, size=n, replace=True)
    bootstrap_means[i] = sample_resample.mean()

rata2_asli = data_layanan.mean()
se_bootstrap = bootstrap_means.std(ddof=1)
ci_bawah, ci_atas = np.percentile(bootstrap_means, [2.5, 97.5])

print(f"Rata-rata sampel asli          : {rata2_asli:.3f} detik")
print(f"Standard error (Bootstrap)      : {se_bootstrap:.3f}")
print(f"95% CI (Bootstrap percentile)   : [{ci_bawah:.3f}, {ci_atas:.3f}] detik")

plt.figure(figsize=(7, 4))
plt.hist(bootstrap_means, bins=40, color="steelblue", edgecolor="white")
plt.axvline(rata2_asli, color="red", linestyle="--", label=f"Rata-rata asli = {rata2_asli:.2f}")
plt.axvline(ci_bawah, color="green", linestyle=":", label="Batas CI 95%")
plt.axvline(ci_atas, color="green", linestyle=":")
plt.title("Distribusi Bootstrap dari Rata-rata Waktu Respon Layanan")
plt.xlabel("Rata-rata (detik)")
plt.ylabel("Frekuensi")
plt.legend()
plt.show()

---
## Soal 2 [40 poin] — Metropolis-Hastings pada MCMC

**Pertanyaan:** Jelaskan proses dan tantangan dalam penggunaan metode Metropolis-Hastings pada MCMC. Mengapa metode ini cocok digunakan dalam Bayesian Statistics?

### Jawaban

**Latar belakang masalah.** Dalam Bayesian Statistics, kita ingin mengetahui distribusi posterior parameter $\theta$ setelah melihat data $D$:
$$p(\theta \mid D) = \frac{p(D \mid \theta)\,p(\theta)}{p(D)}$$
Masalahnya, penyebut $p(D) = \int p(D\mid\theta)p(\theta)\,d\theta$ (disebut *marginal likelihood* atau *evidence*) seringkali berupa integral berdimensi tinggi yang **tidak punya solusi analitik** dan sangat sulit dihitung langsung. MCMC (Markov Chain Monte Carlo) mengatasi ini dengan cara membangun sebuah rantai Markov yang, jika dijalankan cukup lama, distribusi stasionernya sama persis dengan posterior yang kita cari — tanpa perlu menghitung $p(D)$ sama sekali.

**Proses algoritma Metropolis-Hastings:**
1. Mulai dari titik awal $\theta_0$ (tebakan awal parameter).
2. Pada iterasi $t$, usulkan (propose) titik baru $\theta' \sim q(\theta' \mid \theta_t)$ menggunakan **distribusi usulan** (proposal distribution), misalnya $\theta' = \theta_t + \varepsilon$ dengan $\varepsilon \sim N(0, \sigma^2)$ (random walk).
3. Hitung rasio penerimaan:
   $$\alpha = \min\left(1,\ \frac{p(\theta' \mid D)\,q(\theta_t \mid \theta')}{p(\theta_t \mid D)\,q(\theta' \mid \theta_t)}\right)$$
   Karena $p(\theta\mid D) \propto p(D\mid\theta)p(\theta)$, konstanta normalisasi $p(D)$ **saling coret (cancel out)** pada rasio ini — inilah kunci mengapa MH bisa jalan tanpa menghitung integral evidence.
4. Terima $\theta_{t+1} = \theta'$ dengan probabilitas $\alpha$; jika ditolak, tetap di $\theta_{t+1} = \theta_t$.
5. Ulangi ribuan/jutaan kali. Setelah membuang periode awal (*burn-in*), sampel yang tersisa dianggap sampel dari posterior.

**Analogi sehari-hari:** Bayangkan Anda mencari puncak tertinggi di pegunungan pada malam gelap berkabut, hanya bisa merasakan ketinggian tanah tepat di bawah kaki (Anda tidak bisa melihat peta keseluruhan gunung/posterior). Anda melangkah acak ke arah tertentu (proposal): jika tanah di langkah baru **lebih tinggi**, Anda selalu pindah ke sana; jika **lebih rendah**, Anda tetap kadang mau pindah (dengan probabilitas sebanding dengan seberapa jauh lebih rendahnya), supaya tidak terjebak di bukit kecil (*local mode*) dan bisa menjelajah area lain. Setelah berjalan sangat lama, jumlah waktu yang Anda habiskan di setiap titik akan proporsional dengan ketinggian tanah di titik itu — persis seperti sampel MCMC yang "menghabiskan waktu" sebanding dengan densitas posterior di tiap titik.

**Tantangan dalam praktik:**
- **Tuning ukuran langkah (step size / $\sigma$ proposal):** terlalu kecil → hampir semua usulan diterima tapi rantai bergerak sangat lambat (mixing buruk, butuh iterasi sangat banyak). Terlalu besar → banyak usulan ditolak, rantai "macet" di satu titik terlalu lama.
- **Menentukan panjang burn-in:** berapa banyak sampel awal yang harus dibuang sebelum rantai dianggap mencapai distribusi stasioner — tidak ada aturan pasti, biasa dicek lewat *trace plot*.
- **Autokorelasi antar sampel:** sampel MCMC berurutan cenderung berkorelasi (tidak independen), sehingga butuh *thinning* atau diagnostik seperti *effective sample size*.
- **Posterior multimodal:** rantai bisa terjebak di satu mode dan gagal mengeksplorasi mode lain jika langkahnya terlalu kecil.
- **Skalabilitas dimensi tinggi:** semakin banyak parameter, semakin sulit proposal sederhana (random walk) menjelajah ruang parameter secara efisien ("curse of dimensionality").

**Mengapa cocok untuk Bayesian Statistics?** Karena MH **hanya membutuhkan rasio densitas posterior yang tidak ternormalisasi** (unnormalized posterior, yaitu likelihood × prior), bukan nilai posterior yang sudah ternormalisasi penuh. Ini pas sekali dengan struktur masalah Bayesian di mana likelihood × prior mudah dihitung, tapi konstanta normalisasi $p(D)$ nyaris selalu intractable untuk model realistis. Dengan MH, kita bisa melakukan inferensi Bayesian penuh (mendapatkan seluruh bentuk distribusi posterior, bukan cuma titik estimasi) untuk model apa pun asalkan likelihood dan priornya bisa dihitung — termasuk model yang sangat kompleks dan berdimensi tinggi yang mustahil diselesaikan secara analitik.

### Demonstrasi Kode: Metropolis-Hastings untuk Estimasi Rata-rata Posterior

In [ ]:
# --- SOAL 2: Metropolis-Hastings sampler ---
# Kasus: kita punya data yang diasumsikan Normal(mu, sigma^2) dengan sigma diketahui,
# dan kita ingin distribusi posterior dari mu, dengan prior Normal(mu0, tau0^2).
# Kasus ini sengaja dipilih karena punya solusi analitik (conjugate prior), sehingga
# hasil MCMC bisa kita VALIDASI langsung terhadap posterior sejati.

data_obs = np.random.normal(loc=5.0, scale=2.0, size=30)   # data "nyata" yang diamati
sigma = 2.0        # standar deviasi data, dianggap diketahui
mu0, tau0 = 0.0, 10.0   # prior: mu ~ Normal(mu0, tau0^2), prior sangat lebar (uninformative)

def log_posterior(mu, data, sigma, mu0, tau0):
    log_likelihood = np.sum(-0.5 * ((data - mu) / sigma) ** 2)
    log_prior = -0.5 * ((mu - mu0) / tau0) ** 2
    return log_likelihood + log_prior

def metropolis_hastings(data, sigma, mu0, tau0, n_iter=20000, proposal_sd=0.5, mu_start=0.0):
    chain = np.empty(n_iter)
    current_mu = mu_start
    current_logp = log_posterior(current_mu, data, sigma, mu0, tau0)
    n_accept = 0

    for t in range(n_iter):
        proposal_mu = current_mu + np.random.normal(0, proposal_sd)  # proposal simetris (random walk)
        proposal_logp = log_posterior(proposal_mu, data, sigma, mu0, tau0)

        log_alpha = proposal_logp - current_logp  # proposal simetris -> q saling coret
        if np.log(np.random.rand()) < log_alpha:
            current_mu, current_logp = proposal_mu, proposal_logp
            n_accept += 1

        chain[t] = current_mu

    return chain, n_accept / n_iter

chain, accept_rate = metropolis_hastings(data_obs, sigma, mu0, tau0, n_iter=20000, proposal_sd=0.6)

burn_in = 2000
chain_post_burnin = chain[burn_in:]

print(f"Acceptance rate                 : {accept_rate:.2%}")
print(f"Estimasi posterior mean (MCMC)   : {chain_post_burnin.mean():.3f}")

# Validasi terhadap solusi analitik (conjugate Normal-Normal)
n_data = len(data_obs)
posterior_var_analitik = 1 / (1 / tau0**2 + n_data / sigma**2)
posterior_mean_analitik = posterior_var_analitik * (mu0 / tau0**2 + data_obs.sum() / sigma**2)
print(f"Posterior mean analitik (exact)  : {posterior_mean_analitik:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(chain, linewidth=0.5)
axes[0].axvline(burn_in, color="red", linestyle="--", label="akhir burn-in")
axes[0].set_title("Trace Plot Rantai MCMC")
axes[0].set_xlabel("Iterasi")
axes[0].set_ylabel("mu")
axes[0].legend()

axes[1].hist(chain_post_burnin, bins=50, density=True, color="steelblue", label="Sampel MCMC")
axes[1].axvline(posterior_mean_analitik, color="red", linestyle="--", label="Posterior mean analitik")
axes[1].set_title("Distribusi Posterior mu (setelah burn-in)")
axes[1].set_xlabel("mu")
axes[1].legend()
plt.tight_layout()
plt.show()

---
## Soal 3 [30 poin] — Algoritma Iteratif vs Rekursif

**Pertanyaan:** Buatlah perbandingan antara algoritma iteratif dan rekursif dalam konteks komputasi numerik, beserta contoh aplikasinya dalam analisis data.

### Jawaban

| Aspek | Iteratif | Rekursif |
|---|---|---|
| Mekanisme | Menggunakan struktur loop (`for`/`while`) dan variabel state yang diperbarui tiap iterasi | Fungsi memanggil dirinya sendiri dengan subproblem yang lebih kecil |
| Penggunaan memori | Umumnya konstan (O(1) tambahan), tidak menumpuk call stack | Menumpuk *call stack* satu frame per pemanggilan → risiko *stack overflow* untuk rekursi dalam |
| Kecepatan | Biasanya lebih cepat (tidak ada overhead pemanggilan fungsi) | Ada overhead pemanggilan fungsi berulang, kecuali bahasa mendukung *tail-call optimization* (Python tidak mendukungnya) |
| Keterbacaan/kesesuaian masalah | Alami untuk proses yang berulang secara linear (akumulasi, agregasi) | Sangat alami/elegan untuk masalah yang secara struktural rekursif: pohon, divide-and-conquer, definisi matematis rekursif |
| Kapan dipakai | Saat performa & memori penting, atau kedalaman rekursi berpotensi sangat besar | Saat struktur masalah memang rekursif dan kedalaman relatif kecil, demi kejelasan kode |

**Contoh aplikasi dalam analisis data:**

- **Iteratif:** algoritma optimasi numerik seperti *gradient descent* memperbarui parameter model secara iteratif ($\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)$) hingga konvergen; demikian pula *k-means clustering* yang mengulang langkah assign–update secara iteratif; perhitungan statistik kumulatif (running mean/variance) pada data streaming.
- **Rekursif:** algoritma *divide-and-conquer* seperti **Quicksort/Mergesort** untuk mengurutkan data besar (dibutuhkan sebelum menghitung median/kuantil secara efisien); pembangunan struktur pohon rekursif pada **decision tree** atau *dendrogram* pada *hierarchical clustering*; definisi rekursif kombinatorik (misalnya koefisien binomial) yang dipakai dalam perhitungan probabilitas.

**Kapan memilih yang mana:** gunakan rekursi ketika struktur masalah memang alami untuk dipecah menjadi subproblem sejenis dan kedalamannya terbatas (mengutamakan keterbacaan kode) — misalnya traversal pohon. Gunakan iterasi ketika data sangat besar / kedalaman potensial tidak terbatas (menghindari stack overflow) atau ketika performa/memori adalah prioritas, misalnya pemrosesan data dalam jumlah besar di pipeline data streaming.

### Demonstrasi Kode: Perbandingan Kinerja

In [ ]:
# --- SOAL 3a: Fibonacci -- rekursif murni vs iteratif ---
# Menunjukkan bahwa rekursi naif (tanpa memoization) bisa jauh lebih lambat (kompleksitas eksponensial)
# dibanding versi iteratif (kompleksitas linear O(n)).

def fib_rekursif(n):
    if n <= 1:
        return n
    return fib_rekursif(n - 1) + fib_rekursif(n - 2)

def fib_iteratif(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

n_uji = 28

t0 = time.perf_counter()
hasil_rekursif = fib_rekursif(n_uji)
waktu_rekursif = time.perf_counter() - t0

t0 = time.perf_counter()
hasil_iteratif = fib_iteratif(n_uji)
waktu_iteratif = time.perf_counter() - t0

print(f"Fibonacci({n_uji}) rekursif = {hasil_rekursif}, waktu = {waktu_rekursif*1000:.3f} ms")
print(f"Fibonacci({n_uji}) iteratif = {hasil_iteratif}, waktu = {waktu_iteratif*1000:.3f} ms")
print(f"Rekursif naif berjalan sekitar {waktu_rekursif/waktu_iteratif:.0f}x lebih lambat.")

In [ ]:
# --- SOAL 3b: Quicksort (rekursif, divide-and-conquer) vs sort() bawaan Python (iteratif/Timsort) ---
# Aplikasi nyata dalam analisis data: mengurutkan data sebelum menghitung median/kuantil.

def quicksort_rekursif(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    kiri = [x for x in arr if x < pivot]
    tengah = [x for x in arr if x == pivot]
    kanan = [x for x in arr if x > pivot]
    return quicksort_rekursif(kiri) + tengah + quicksort_rekursif(kanan)

data_besar = list(np.random.randint(0, 100000, size=5000))

t0 = time.perf_counter()
hasil_quicksort = quicksort_rekursif(data_besar.copy())
waktu_quicksort = time.perf_counter() - t0

t0 = time.perf_counter()
hasil_builtin = sorted(data_besar.copy())
waktu_builtin = time.perf_counter() - t0

assert hasil_quicksort == hasil_builtin, "Hasil pengurutan harus sama"

median_manual = hasil_quicksort[len(hasil_quicksort) // 2]
print(f"Quicksort rekursif : {waktu_quicksort*1000:.2f} ms")
print(f"sorted() bawaan    : {waktu_builtin*1000:.2f} ms  (implementasi Timsort, pada dasarnya iteratif)")
print(f"Median hasil pengurutan (n={len(data_besar)}): {median_manual}")
print("\nCatatan: sorted() bawaan Python jauh lebih cepat karena teroptimasi di level C,")
print("namun secara konsep quicksort di atas menunjukkan bagaimana pendekatan REKURSIF")
print("divide-and-conquer bekerja: pecah data, urutkan tiap bagian secara rekursif, gabungkan.")

---
## Kesimpulan

Ketiga soal di atas menggambarkan bagaimana komputasi statistik modern (Bootstrap, MCMC/Metropolis-Hastings, dan pemilihan strategi algoritma iteratif/rekursif) sangat bergantung pada daya komputasi yang tersedia sejak era PC. Tanpa kemampuan mengulang simulasi ribuan hingga jutaan kali secara cepat, metode-metode ini hanya akan menjadi teori tanpa bisa diterapkan pada data nyata.

**Referensi materi:**
1. How Computational Statistics Became the Backbone of Modern Data Science
2. Numerical Foundations for Modern Statistical Analysis